<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap6_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
６章言語モデルのファインチューニング

- ファインチューニング済みエンコーダーモデルを用いたテキストのトピック分類
- 現代の LLM 時代におけるエンコーダーベースモデルの役割の理解
- デコーダーモデルを使った特定のスタイルのテキスト生成
- 命令型ファインチューニングによる単一モデルでの複数タスクの解決
- 小さいGPUでもモデルを訓練できるパラメーター効率の高いファインチューニング手法
- よりすくに計算資源でモデルの推論を実行できる手法

6.1.1 データセットの特定

In [1]:
from datasets import load_dataset

#az news データセットはテキスト分類モデルのベンチマークやデータマイニング、情報検索、データストリーミングなどの研究で広く用いられている。
# 訓練用のサンプルは１２万件あり、ファインにチューニングには十分
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [2]:
# データの具体的な例を見ていこう

raw_train_datasets = raw_datasets["train"]
raw_train_datasets[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

//{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

サンプルにテキストとラベルが含まれているが、２はどのクラスを指しているのか？
これを知るにはデータセットの features とその label フィールドを見ればいい。

In [3]:
print(raw_train_datasets.features)

# results
# {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
# 0なら世界のニュース、1ならスポーツ、2ならビジネス、３なら科学技術のニュース

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


# 6.1.2 使用するモデルタイプの定義

## Transformer おさらい

- エンコーダーモデル：入力の意味表現を捉える
- デコーダーモデル：文章などの新しいシーケンスを出力することを目的にしたモデル。テキスト生成に最適。
- エンコーダーデコーダー型モデル：入力シーケンスを異なる出力シーケンスに変換するタスクに適している

今回は、**分類ヘッド付きエンコーダーモデル**をアプローチとして採用する。
エンコーダーモデルにシンプルな分類ネットワーク（ヘッド）を埋め込みに追加してファインチューニングする方法。

ベースモデルの要件は以下の４つ

- エンコーダベースであること
- GPU を使えば数分くらいでファインチューニングできるモデル
- 事前訓練で確かな成果を残しているもの
- 短いテキストシーケンスを処理できること

DistilBERT が良さげらしい。